In [34]:
import pandas as pd
import altair as alt

In [24]:
file_path = '/Users/juanpazmino/Documents/Desarrollo/Personal_projects/data_visualization/Lab01/01mdi_homicidios_intencionales_pm_2014_2025.xlsx'
df = pd.read_excel(file_path, sheet_name='1', header=1)
df = df.drop(df.columns[0], axis=1)
df.head()

,tipo_muerte,zona,subzona,distrito,circuito,codigo_subcircuito,subcircuito,codigo_provincia,provincia,codigo_canton,...,medida_edad,sexo,genero,etnia,estado_civil,nacionalidad,discapacidad,profesion_registro_civil,instruccion,antecedentes
0,ASESINATO,ZONA 1,ESMERALDAS,ESMERALDAS,LAS PALMAS,08D01C02S01,LAS PALMAS 1,8,ESMERALDAS,801,...,A,MUJER,FEMENINO,AFRO,SOLTERO,ECUADOR,NINGUNA,ESTADO PERSONAL,SIN_DATO,SIN_DATO
1,ASESINATO,ZONA 4,MANABÍ,PORTOVIEJO,SAN PABLO,13D01C05S02,SAN PABLO 2,13,MANABÍ,1301,...,A,HOMBRE,MASCULINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,TRABAJADOR GENERAL,BASICA,SIN_DATO
2,ASESINATO,ZONA 4,MANABÍ,PORTOVIEJO,SAN PABLO,13D01C05S02,SAN PABLO 2,13,MANABÍ,1301,...,A,HOMBRE,MASCULINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,MAESTRO DE OBRA/CONSTRUCCIÓN,SIN_DATO,SIN_DATO
3,ASESINATO,ZONA 4,MANABÍ,MANTA,LA PILA,13D02C14S01,LA PILA 1,13,MANABÍ,1309,...,A,MUJER,FEMENINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,SIN_DATO,SIN_DATO,SIN_DATO
4,ASESINATO,ZONA 4,MANABÍ,MANTA,LA PILA,13D02C14S01,LA PILA 1,13,MANABÍ,1309,...,A,MUJER,FEMENINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,ESTADO PERSONAL,SECUNDARIA,SIN_DATO


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39773 entries, 0 to 39772
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   tipo_muerte               39773 non-null  str           
 1   zona                      39773 non-null  str           
 2   subzona                   39773 non-null  str           
 3   distrito                  39773 non-null  str           
 4   circuito                  39773 non-null  str           
 5   codigo_subcircuito        39773 non-null  str           
 6   subcircuito               39773 non-null  str           
 7   codigo_provincia          39773 non-null  int64         
 8   provincia                 39773 non-null  str           
 9   codigo_canton             39773 non-null  int64         
 10  canton                    39773 non-null  str           
 11  coordenada_y              39773 non-null  str           
 12  coordenada_x              397

In [30]:
date_str = pd.to_datetime(df['fecha_infraccion'], errors='coerce').dt.strftime('%Y-%m-%d')
time_str = df['hora_infraccion'].astype(str)

df['fecha_hora_infraccion'] = pd.to_datetime(
    date_str + ' ' + time_str,
    errors='coerce' 
)

df = df.drop(columns=['fecha_infraccion', 'hora_infraccion'])

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39773 entries, 0 to 39772
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   tipo_muerte               39773 non-null  str           
 1   zona                      39773 non-null  str           
 2   subzona                   39773 non-null  str           
 3   distrito                  39773 non-null  str           
 4   circuito                  39773 non-null  str           
 5   codigo_subcircuito        39773 non-null  str           
 6   subcircuito               39773 non-null  str           
 7   codigo_provincia          39773 non-null  int64         
 8   provincia                 39773 non-null  str           
 9   codigo_canton             39773 non-null  int64         
 10  canton                    39773 non-null  str           
 11  coordenada_y              39773 non-null  str           
 12  coordenada_x              397

# **Pregunta 1: Patrón temporal semanal**

¿Existen días de la semana donde ocurren más homicidios?
¿Se observa concentración en fines de semana o días laborales?

**Enfoque analítico esperado:**
Identificar periodicidad semanal y comparar intensidad por día.

In [ ]:
# 1. Extraer el día de la semana numérico (0 = Lunes, 6 = Domingo)
df['dia_semana_num'] = df['fecha_hora_infraccion'].dt.dayofweek

# 2. Mapear a nombres de días para facilitar la lectura
dias_map = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles',
    3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}
df['dia_semana'] = df['dia_semana_num'].map(dias_map)

# 3. Conteo de homicidios por día de la semana (garantizando el orden correcto de los días)
contagem_por_dia = df['dia_semana'].value_counts().reindex(list(dias_map.values()))
print("--- Conteo de Homicidios por Día de la Semana ---")
print(contagem_por_dia)

# 4. Agrupar los datos comparando Fin de Semana vs Día Laboral
df['tipo_dia'] = df['dia_semana_num'].apply(lambda x: 'Fin de Semana' if x >= 5 else 'Día Laboral')
contagem_tipo_dia = df['tipo_dia'].value_counts()
print("\n--- Conteo por Tipo de Día (Fin de Semana vs Día Laboral) ---")
print(contagem_tipo_dia)

# 5. Visualización gráfica de los resultados con Altair
df_plot = contagem_por_dia.reset_index()
df_plot.columns = ['Día de la Semana', 'Número de Ocurrencias']

chart = alt.Chart(df_plot).mark_bar().encode(
    x=alt.X('Día de la Semana:N', 
            sort=list(dias_map.values()), 
            title='Día de la Semana',
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('Número de Ocurrencias:Q', title='Número de Ocurrencias'),
    color=alt.Color('Día de la Semana:N', legend=None, scale=alt.Scale(scheme='viridis')),
    tooltip=['Día de la Semana', 'Número de Ocurrencias']
).properties(
    title='Distribución de Homicidios por Día de la Semana',
    width=600,
    height=400
)

chart.show()


--- Conteo de Homicidios por Día de la Semana ---
dia_semana
Lunes        5313
Martes       4891
Miércoles    4830
Jueves       5023
Viernes      5577
Sábado       6599
Domingo      7515
Name: count, dtype: int64

--- Conteo por Tipo de Día (Fin de Semana vs Día Laboral) ---
tipo_dia
Día Laboral      25659
Fin de Semana    14114
Name: count, dtype: int64


alt.Chart(...)